# Chapter 4 (part 5) — Polynomial, Spline, and Generalized Additive Models

Covers the last Ch 4 section: Polynomial and Spline Regression → Generalized Additive Models (GAM), on Ames Housing.

The thread: **"linear regression" means linear in the *coefficients*, not in the raw variables.** You can feed it `x`, `x²`, `log(x)`, or a set of bendy basis functions, and it's still ordinary least squares. The only real question is how much flexibility to allow: too little and you miss real curvature (bias); too much and you chase noise (variance). Polynomials add flexibility *globally* (every term affects the whole curve, which makes the tails wild). Splines add it *locally* (cut the range into pieces joined smoothly at knots). A GAM gives each predictor its own spline and adds them up — more flexible than a line, more interpretable than a black box.

Tooling notes (nothing new to install):
- Polynomials: `I(x**2)` inside a `smf.ols` formula. Beware: raw powers of big numbers (e.g., `Year_Built**3`) are numerically nasty; center and scale first, or use orthogonal polynomials.
- Splines: patsy's `bs(x, df=..., degree=3)` (B-spline) and `cr(x, df=...)` (natural cubic regression spline) work inside formulas. **Gotcha:** `bs()` can't predict for points outside the range of the data it was fit on, so on a train/test split pass `lower_bound=` and `upper_bound=` computed from the *full* column.
- GAM: `statsmodels.gam.api.GLMGam` with `BSplines` (it adds a smoothness penalty so you can tune wiggliness with `alpha`). The book uses R's `mgcv`; there's no `pygam` here.

Ask for a hint if you get stuck; the first couple of asks per question get a nudge, not the answer.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from patsy import dmatrix
from statsmodels.gam.api import GLMGam, BSplines
from sklearn.model_selection import train_test_split, KFold

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
ames.shape

## 1. Polynomial Regression

A quadratic adds an `x²` column: `y ≈ b0 + b1*x + b2*x²`. The slope is no longer a constant. Numeric picture: if `b1 = 100` and `b2 = 0.02`, the extra price of one more unit of x is about `100 + 2*0.02*x`, which is 100 + 40 = $140 when x = 1,000 and 100 + 80 = $180 when x = 2,000. The single "slope" doesn't exist any more; there's a slope *at each x*, and `b1` alone is just the slope at x = 0 (which might be outside your data).

Another numeric trap: `Year_Built` is around 1,950, so `Year_Built²` is around 3.8 million and the two columns are nearly perfectly correlated (see the multicollinearity section last notebook). Centering first (`x - x.mean()`) takes most of that away without changing the fitted curve.

**Question 1.** (a) Fit `Sale_Price ~ Year_Built + I(Year_Built**2)` as is, then again with a centered `Year_Built`. Compare the coefficient tables and VIFs; confirm that the fitted values are identical. (b) Compute the marginal effect of an extra year of age at 1920, 1960, and 2000 for the centered model (the derivative of the fitted curve). Where does the curve turn? (c) Plot the data, the straight-line fit, and the quadratic fit. Does the quadratic capture what you saw in the partial-residual plot from the diagnostics notebook? (d) Which of the two models would you believe at `Year_Built = 2020`?

*Tools for this question:*
- `I(x**2)` inside the formula protects the arithmetic; centered version: create `df["yb_c"] = df["Year_Built"] - df["Year_Built"].mean()` first.
- Marginal effect of a quadratic `b1*x + b2*x**2` is `b1 + 2*b2*x`; get the values with `res.params[...]`.

_Your interpretation:_

## 2. How Flexible Should the Curve Be?

More terms always fit the training data better — the question is held-out performance. Numeric picture: with 5 points, a degree-4 polynomial passes through all of them exactly (zero training error) and swings wildly between them; a line misses all five a little but is stable.

**Question 2.** (a) For `Gr_Liv_Area` (alone as predictor, to keep the picture clean), fit polynomials of degree 1 through 10 using 5-fold cross-validation (`KFold`). Record train RMSE and CV RMSE for each degree. Plot both curves. (If statsmodels complains about a singular matrix at higher degrees, that is itself the lesson from the multicollinearity notebook: standardize `Gr_Liv_Area` first.) (b) Which degree would you pick, and what rule of thumb did you use (lowest CV, or simplest within one standard error of the lowest)? (c) Plot the fitted curve for degrees 2, 5, and 10 over `Gr_Liv_Area` from 300 to 7,000 (beyond the 5,642 maximum). What happens at the edges and beyond the data? (d) Repeat the experiment with only 200 random homes as the training data. How does the best degree change, and why?

*Tools for this question:*
- `KFold(n_splits=5, shuffle=True, random_state=0).split(df)` yields `(train_idx, test_idx)` arrays — use `df.iloc[idx]`.
- Degree-k formula: `"Sale_Price ~ " + " + ".join(["z"] + [f"I(z**{j})" for j in range(2, k + 1)])` where `z` is the standardized column `(x - x.mean()) / x.std()`.

_Your interpretation:_

## 3. Splines

A **spline** cuts the x-axis into pieces at **knots** and fits a separate low-degree polynomial in each piece, constrained to join smoothly. Numeric picture: put knots at 1,000 and 2,000 sq ft. Below 1,000 the curve is one cubic, between 1,000 and 2,000 another, above 2,000 a third — but all three share the same value, slope, and curvature where they meet, so there are no kinks. A change in the shape at 2,000 doesn't disturb the fit below 1,000, which is the key difference from a polynomial. The number of parameters (the **degrees of freedom**, `df`) is the knob: more knots, more flexibility.

A **natural** spline additionally forces the curve to be a straight line outside the outermost knots, which tames the tails.

**Question 3.** Use `Gr_Liv_Area` as the single predictor. (a) Fit `bs(Gr_Liv_Area, df=k)` for k = 4, 6, 8, 12 and plot the fits together. Where do the knots sit by default (hint: quantiles; check `dmatrix` or the patsy docs)? (b) Fit `cr(Gr_Liv_Area, df=6)` (natural) and compare with `bs(..., df=6)` at the edges, especially for large homes. Which behaves more sensibly past 4,000 sq ft, and what does each assume? (c) Compare held-out RMSE (5-fold CV or a hold-out split; remember `lower_bound` / `upper_bound`) for the best polynomial from Question 2 and splines with 4-12 df. (d) Place two knots by hand at 1,000 and 2,000 (`knots=[1000, 2000]`) and compare to the default placement. When would you place knots by hand instead of by quantiles?

*Tools for this question:*
- `bs(x, df=6, degree=3, lower_bound=lo, upper_bound=hi)` B-spline basis; `cr(x, df=6)` natural cubic spline; `knots=[1000, 2000]` places interior knots by hand (use instead of `df`).
- Inspect the basis: `patsy.dmatrix("bs(Gr_Liv_Area, df=6)", df, return_type="dataframe")`.

_Your interpretation:_

## 4. Generalized Additive Models

A GAM generalizes the idea: `y ≈ b0 + f1(x1) + f2(x2) + b3*x3`, where each `f` is a smooth curve learned from the data (typically a spline with a **smoothness penalty** that discourages wiggliness; a bigger penalty means a flatter, more line-like curve). The effects still *add*, so you can plot each `f` on its own and read it like a coefficient, "the contribution of living area to price, holding the other terms fixed." The penalty replaces hand-tuning the number of knots: use lots of knots and let the penalty decide how much wiggle to keep.

**Question 4.** Fit `GLMGam` with `BSplines` on `Year_Built` and `Gr_Liv_Area` (say `df=[8, 8]`, `degree=[3, 3]`), a linear term for `Garage_Area` and `Total_Bsmt_SF`, and `alpha=[1, 1]` as a first guess. (a) Plot the partial effect of each smooth (`res.plot_partial(i, cpr=True)`). Which one is closest to straight? Which shows the clearest bend? (b) Raise `alpha` to a very large value (like 1e6) and then lower it to something tiny (like 1e-3), re-plotting each time. What happens to the curves? Relate this to the polynomial-degree and spline-df knobs from the last two sections. (c) Use `gam.select_penweight` or a manual hold-out grid to pick `alpha`. (d) Compare held-out RMSE of the GAM to the straight-line model on the same four predictors. How large is the improvement? Is it consistent with how curved the plots were? (e) A GAM assumes additivity. Add the product `Year_Built * Gr_Liv_Area` to the residuals check (correlate the GAM's residuals with `Year_Built * Gr_Liv_Area` after centering both, or fit a small model of the residuals on that product). Is there leftover structure?

*Tools for this question:*
- `bs = BSplines(df[["Year_Built", "Gr_Liv_Area"]], df=[8, 8], degree=[3, 3])` builds the smoothers.
- `GLMGam.from_formula("Sale_Price ~ Garage_Area + Total_Bsmt_SF", data=df, smoother=bs, alpha=[1, 1])` then `.fit()`; `alpha` is the smoothness penalty per smoother (bigger = flatter).
- `res.plot_partial(i, cpr=True)` plots smooth number `i`; `gam.select_penweight(criterion="gcv")` (call `.fit()` once first) returns a tuple whose first element is the chosen `alpha` array.

_Your interpretation:_

## 5. Which Tool When? (interview prep)

**Question 5.** Using Ames, compare on one held-out split the following models for `Sale_Price`, all on `Gr_Liv_Area, Year_Built, Garage_Area, Total_Bsmt_SF`: (i) straight line; (ii) straight line on `log(Sale_Price)` with a back-transform (use the *smearing* adjustment `np.exp(pred) * mean(np.exp(resid))` so you don't systematically under-predict); (iii) log-log; (iv) GAM from Question 4. Then, in a markdown cell, write the three-sentence answer you'd give to: "When would you reach for a transformation vs. a spline/GAM vs. a tree-based model?" Include one reason a log transform can be preferable to a spline even when the spline fits slightly better.

*Tools for this question:*
- Smearing back-transform: `np.exp(res.predict(test)) * np.mean(np.exp(res.resid))`; `np.log1p(x)` is `log(1+x)`, safe when `x` can be 0.

_Your answer:_

---
# Exam — check your intuition

7 questions. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, a few bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Polynomial vs. spline: reading the scoreboard
*Role lean: DS · ML*

One predictor, `Lot_Area`, with a very skewed distribution (a handful of enormous lots).

```text
Fit on 2051 training homes; scored on 879 held-out homes. Predictor is Lot_Area only (range 1,300 to 215,245 sq ft; median 9,436; 35 homes above 30,000).

                           model  train RMSE  test RMSE rank warning
       straight line in Lot_Area       77689      75440             
            polynomial, degree 2       75797      72583             
            polynomial, degree 3       74813      71512             
            polynomial, degree 5       81969      78125          yes
            polynomial, degree 8       83903      79408          yes
    log(Lot_Area), straight line       75116      72671             
     B-spline, 5 df, on Lot_Area       74736      71678             
B-spline, 5 df, on log(Lot_Area)       72340      69675
```

(a) Adding a term to a least-squares fit can never make the *training* error go up — yet look at the training RMSE at degrees 5 and 8. What's going on, and what does the "rank warning" column tell you? (Think about how big and how correlated the columns `Lot_Area`, `Lot_Area^2`, ..., `Lot_Area^8` are, and what a computer does with numbers like 215,245^8.) What would you do to fix it? (b) Between degree 2 and degree 3 the held-out error improves only a little. Using the table, what would you pick and why would "simplest model that's about as good as the best" be a reasonable rule? (c) The 5-df spline on `log(Lot_Area)` beats the 5-df spline on raw `Lot_Area`. Why would transforming first help a spline that can already bend? (d) The straight line in `log(Lot_Area)` has a single slope yet nearly matches the polynomial. What does that say about the *shape* of the true relationship?

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — What happens off the edge of the data
*Role lean: ML · DS · interview*

```text
Predicted Sale_Price at a few lot sizes, models fit on all 2,930 homes (largest lot in the data: 215,245):

 Lot_Area (sq ft)  cubic polynomial  straight line in log(Lot_Area)
             1300            113178                           71125
            10000            184003                          187536
            50000            322523                          279368
           100000            236644                          318918
           215245            466864                          362660
           300000           2669997                          381603

For reference, the most expensive home in the data sold for 755,000.
```

(a) Which model's behavior at 300,000 sq ft is plainly broken, and what property of polynomials causes it? (b) The log model looks sensible there. Is it actually *right* at 300,000 sq ft, or just not obviously wrong? What would you need to know to trust any model out there? (c) A natural cubic spline (patsy's `cr()`) is constrained to be *linear* beyond the boundary knots, whereas an ordinary B-spline (`bs()`) in patsy refuses to predict outside the range it was fit on. How does each choice trade safety against honesty about extrapolation? (d) If this were a production pricing model, what guardrail would you add around features like lot size?

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — Interpreting a quadratic term
*Role lean: DS · Causal · interview*

```text
Sale_Price ~ Garage_Area + Garage_Area^2
                           coef    std err        t   P>|t|
Intercept            83675.2614  3973.8758  21.0563  <0.001
Garage_Area            162.4252    15.2414  10.6568  <0.001
I(Garage_Area ** 2)      0.0755     0.0144   5.2567  <0.001

n = 2930   R-squared = 0.415   Adj. R-squared = 0.415   Residual std error = 61,107
(Garage_Area ranges from 0 to 1,488 sq ft; median 480.)
```

(a) The squared term is highly significant and positive. Write the formula for "the extra price of one more square foot of garage" at a given `Garage_Area`, and evaluate it at 300 sq ft and 900 sq ft. (b) A stakeholder asks "so what's the effect of garage area?" Why is a single number the wrong answer? (c) The quadratic's turning point is at a *negative* garage area. What does that tell you about the shape over the range that actually exists? (d) Is this a *nonlinear* model in the sense of "can't be fit by linear regression"? Explain what "linear" refers to in "linear regression." 

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — Bias, variance, and the number of knots
*Role lean: ML · DS · interview*

```text
Spline of Year_Built (the only predictor) with k degrees of freedom. Each fit uses only 150 random homes; scored on the other 2,780; averaged over 25 random draws.

 spline df  train RMSE  held-out RMSE
         3       61895          63728
         5       61479          64622
         8       61008          65653
        12       60608          70995
        16       60458          84182
```

(a) Describe both columns in words: where is the sweet spot, and why does train RMSE keep dropping while held-out RMSE rises steadily? Which of bias or variance dominates at df = 3, and which at df = 16? (b) With all 2,930 homes in the training set instead of 150, would you expect the best df to move up, down, or stay? Why? (c) Interview version: "How do you choose the number of knots (or the polynomial degree, or the smoothing penalty)? And how is that the same question as choosing the depth of a decision tree?" 

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — A GAM is additive: what does that buy and cost?
*Role lean: DS · ML*

```text
Predicted price for an otherwise-median home (median Gr_Liv_Area = 1442; median Year_Remod_Add = 1993 in the second table; median Total_Bsmt_SF = 990 in the first):

 Year_Remod_Add  homes with that value  additive spline  linear model
           1950                    361           134532        135517
           1960                     29           144056        146390
           1970                     44           146132        157262
           1980                     30           155677        168135
           1990                     29           166405        179007
           2000                    104           175014        189880
           2005                    141           188689        195316
           2010                     13           231824        200752

 Total_Bsmt_SF  homes with at least this much  additive spline  linear model
             0                           2930           136204        121151
           500                           2753           147536        152019
          1000                           1436           169287        182887
          1500                            431           211775        213754
          2000                             58           262667        244622
          3000                              6           343689        306358
          4000                              2           333318        368093
```

(a) The `Year_Remod_Add` column has 361 homes with the value 1950 (the column is floored at 1950: 339 of those 361 are homes built *before* 1950, whose remodel year is recorded as 1950 regardless of what happened). Describe the spline's curve for `Year_Remod_Add` in plain words, and say what that pile-up at 1950 does to how you should read the left end of the curve (what is the 1950 group really made of?). (b) Compare the spline with the linear model for `Total_Bsmt_SF` at 0, 3,000, and 4,000 sq ft. Where do they differ most, and how many homes are out there — which one are you more worried about, and why? (c) A GAM gives each predictor its own curve but makes the effects *add up*. Give a concrete example from housing of an interaction that an additive model cannot express, and say what you'd add to capture it. (d) What do you give up in interpretability moving from the linear model to the GAM, and what do you keep?

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_

### Exam Q6 — Does the nonlinearity pay for itself?
*Role lean: ML · DS · interview*

```text
Held-out RMSE (2,051 train / 879 test), same three predictors in every model:
  linear (3 predictors)                                             43,117
  log-log version (back-transformed)                                47,177
  additive spline (6 df each on Year_Remod_Add, Total_Bsmt_SF)      40,575
  gradient boosting (same 3 predictors)                             41,658
```

(a) The additive spline beats the straight line by about $2.5k RMSE on held-out data, while the log-log version is *worse* than the straight line. Give a reason each result could be true. (Think about what each transformation assumes about the shape and about the spread of errors, and about the back-transformation from log dollars to dollars.) (b) Gradient boosting lands between the additive spline and the linear model. What kinds of structure can it capture that the additive spline can't, and what does it mean that it does *not* beat the spline here? (c) You're asked whether to ship the additive spline or the boosting model. List the questions you'd ask about the business use before deciding, and one scenario for each choice. (d) How would you decide whether the ~$2.5k improvement is real rather than a quirk of this particular 70/30 split?

**Your answer (Q6):**

> _write here_

**Review (Q6):**

> _pending — filled in after you answer_

### Exam Q7 — Pick the right tool
*Role lean: DS · Causal · ML*

For each situation, choose among: straight line, log transform, polynomial, spline / GAM, or tree-based model — and give a one- or two-sentence justification: (1) a regulator wants a defensible statement about how price changes per additional bathroom; (2) a pricing team wants the best possible predictions of sale price from 80 mixed features and doesn't care about interpretation; (3) you're doing an analysis for a decision-science team where the effect of building age on price clearly bends, and you need to show the shape to executives; (4) you plan to use the fitted model to *extrapolate* year-built effects to homes built in the future. For (4), say what's wrong with each of your instinctive choices.

**Your answer (Q7):**

> _write here_

**Review (Q7):**

> _pending — filled in after you answer_